# Case Study — ScoutMetrics · Football Scouting Analytics
**Project (context):** ScoutMetrics (personal sports-analytics lab) · **Synthetic tournament data for portfolio demo**

This notebook mirrors the **ScoutMetrics** workflow used to explore World Cup–style master files:
team key stats, phases of play, player line breaks, and physical load — then turns them into
**scouting scores** coaches and analysts can compare across matches.

**Why this notebook matters**  
Scouting decisions improve when teams share the same definitions of *progression*, *physical output*,
and *phase dominance*. This case study shows how multi-table match data becomes a ranked player board.

> All figures are **fictional** and seeded for reproducibility (`SEED = 26`).  
> Methodology inspired by the ScoutMetrics Mundial 2026 moneyball master-file structure.


## 0. Setup

Fix reproducibility and export paths so the analysis can be rerun from `notebooks/` or the portfolio root.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 26
rng = np.random.default_rng(SEED)

ROOT = Path.cwd()
if (ROOT / "data").exists():
    DATA_DIR = ROOT / "data"
elif (ROOT.parent / "data").exists():
    DATA_DIR = ROOT.parent / "data"
else:
    DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = DATA_DIR / "scoutmetrics_player_scores.csv"
print("Data folder:", DATA_DIR.resolve())
print("Will export:", OUT_CSV.name)

## 1. Synthetic tournament tables

We recreate the **core ScoutMetrics sheets** at small scale:
- `matches` — schedule / results
- `team_stats` — long-format team metrics
- `phases` — % of play by phase
- `linebreaks` / `physical` — player-level progression & load


In [ ]:
teams = [
    "Spain", "France", "Argentina", "England", "Brazil", "Germany",
    "Portugal", "Netherlands", "Colombia", "Japan", "USA", "Morocco",
]
n_matches = 48
matches = pd.DataFrame({
    "match_no": range(1, n_matches + 1),
    "group": rng.choice(list("ABCDEFGH"), n_matches),
    "team1": rng.choice(teams, n_matches),
    "team2": rng.choice(teams, n_matches),
})
# avoid same team twice
mask = matches["team1"].eq(matches["team2"])
while mask.any():
    matches.loc[mask, "team2"] = rng.choice(teams, mask.sum())
    mask = matches["team1"].eq(matches["team2"])
matches["team1_goals"] = rng.integers(0, 4, n_matches)
matches["team2_goals"] = rng.integers(0, 4, n_matches)
matches.head()

In [ ]:
metrics = [
    "possession_pct", "shots", "shots_on_target", "passes_completed",
    "xg", "pressures", "line_breaks_team",
]
rows = []
for _, m in matches.iterrows():
    for side, team in (("home", m["team1"]), ("away", m["team2"])):
        base = {
            "match_no": m["match_no"],
            "group": m["group"],
            "team": team,
            "team_side": side,
        }
        vals = {
            "possession_pct": round(float(rng.uniform(38, 62)), 1),
            "shots": int(rng.integers(4, 22)),
            "shots_on_target": int(rng.integers(1, 10)),
            "passes_completed": int(rng.integers(220, 620)),
            "xg": round(float(rng.uniform(0.2, 2.8)), 2),
            "pressures": int(rng.integers(80, 220)),
            "line_breaks_team": int(rng.integers(8, 40)),
        }
        for metric, value in vals.items():
            rows.append({**base, "metric": metric, "value": value})

team_stats = pd.DataFrame(rows)
# normalize possession so home+away ≈ 100
poss = team_stats[team_stats["metric"].eq("possession_pct")].copy()
# leave as-is for demo simplicity
team_stats.head()

In [ ]:
phases_labels = ["Build-up", "Progression", "Final third", "Defensive block", "Transition"]
phase_rows = []
for _, m in matches.iterrows():
    for team in (m["team1"], m["team2"]):
        weights = rng.dirichlet(np.ones(len(phases_labels)) * 2.0)
        for phase, pct in zip(phases_labels, weights):
            phase_rows.append({
                "match_no": m["match_no"],
                "group": m["group"],
                "team": team,
                "phase": phase,
                "percentage": round(float(pct * 100), 1),
            })
phases = pd.DataFrame(phase_rows)
phases.head()

In [ ]:
# Player-level line breaks + physical load (≈ 11 players × 2 teams × matches subsample)
sample_matches = matches.sample(20, random_state=SEED)
player_rows = []
phys_rows = []
for _, m in sample_matches.iterrows():
    for team in (m["team1"], m["team2"]):
        for shirt in range(1, 12):
            attempted = int(rng.integers(0, 18))
            completed = int(rng.integers(0, attempted + 1)) if attempted else 0
            player_rows.append({
                "match_no": m["match_no"],
                "team": team,
                "shirt": shirt,
                "player": f"{team[:3].upper()}_{shirt:02d}",
                "line_breaks_attempted": attempted,
                "line_breaks_completed": completed,
                "line_break_completion_pct": round(100 * completed / attempted, 1) if attempted else 0.0,
                "through": int(rng.integers(0, max(completed, 1))),
                "around": int(rng.integers(0, max(completed, 1))),
                "over": int(rng.integers(0, max(completed, 1))),
            })
            dist = float(rng.uniform(6500, 12500))
            phys_rows.append({
                "match_no": m["match_no"],
                "team": team,
                "shirt": shirt,
                "player": f"{team[:3].upper()}_{shirt:02d}",
                "total_distance_m": round(dist, 0),
                "high_speed_runs_z3": int(rng.integers(8, 55)),
                "sprints_z4_z5": int(rng.integers(2, 28)),
                "top_speed_kmh": round(float(rng.uniform(27, 35)), 1),
            })

linebreaks = pd.DataFrame(player_rows)
physical = pd.DataFrame(phys_rows)
print(linebreaks.shape, physical.shape)
linebreaks.head()

## 2. Team KPI panel

Pivot long team metrics and rank sides by xG and progressive line breaks.


In [ ]:
team_wide = (
    team_stats.pivot_table(
        index=["match_no", "group", "team", "team_side"],
        columns="metric",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)
team_rank = (
    team_wide.groupby("team", as_index=False)
    .agg(
        matches=("match_no", "nunique"),
        avg_xg=("xg", "mean"),
        avg_shots=("shots", "mean"),
        avg_line_breaks=("line_breaks_team", "mean"),
        avg_possession=("possession_pct", "mean"),
    )
    .sort_values("avg_xg", ascending=False)
)
team_rank["avg_xg"] = team_rank["avg_xg"].round(2)
team_rank.head(10)

## 3. Phases of play

Which teams spend more time in **Final third** vs **Defensive block**?


In [ ]:
phase_team = (
    phases.groupby(["team", "phase"], as_index=False)["percentage"].mean()
)
final_third = (
    phase_team[phase_team["phase"].eq("Final third")]
    .sort_values("percentage", ascending=False)
    .head(10)
    .rename(columns={"percentage": "avg_final_third_pct"})
)
final_third

## 4. Player scouting score

Combine line-break completion, volume, distance, and sprint volume into a 0–100 scouting index.


In [ ]:
players = linebreaks.merge(
    physical,
    on=["match_no", "team", "shirt", "player"],
    how="inner",
)

agg = players.groupby(["team", "player"], as_index=False).agg(
    matches=("match_no", "nunique"),
    lb_attempted=("line_breaks_attempted", "sum"),
    lb_completed=("line_breaks_completed", "sum"),
    avg_distance_m=("total_distance_m", "mean"),
    avg_sprints=("sprints_z4_z5", "mean"),
    avg_top_speed=("top_speed_kmh", "mean"),
)
agg["lb_completion_pct"] = np.where(
    agg["lb_attempted"] > 0,
    (100 * agg["lb_completed"] / agg["lb_attempted"]).round(1),
    0.0,
)

# Min-max components → scouting score
def minmax(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(50.0, index=s.index)
    return 100 * (s - lo) / (hi - lo)

agg["score"] = (
    0.30 * minmax(agg["lb_completed"])
    + 0.25 * minmax(agg["lb_completion_pct"])
    + 0.25 * minmax(agg["avg_distance_m"])
    + 0.20 * minmax(agg["avg_sprints"])
).round(1)

board = agg.sort_values("score", ascending=False).reset_index(drop=True)
board.head(15)

In [ ]:
# Export board for BI / case-study download
board.to_csv(OUT_CSV, index=False)
print("Exported:", OUT_CSV.resolve())
print("Players scored:", len(board))
print("Top scout score:", board.iloc[0][["player", "team", "score"]].to_dict())

## 5. Takeaways

| Layer | Output |
|-------|--------|
| Team | Ranked xG / progression panel |
| Phases | Final-third dominance by team |
| Player | Scouting score blending progression + physical |

This is the same **multi-table → KPI → ranking** pattern used in ScoutMetrics when reading
Moneyball-style World Cup master files (team stats, phases, line breaks, physical).


In [ ]:
summary = {
    "matches_modeled": int(matches["match_no"].nunique()),
    "teams": int(team_rank.shape[0]),
    "players_scored": int(board.shape[0]),
    "top_player": board.iloc[0]["player"],
    "top_score": float(board.iloc[0]["score"]),
    "export": str(OUT_CSV.name),
}
summary